# Graph scoring without gold labels

Whether a change to the extraction prompt helped cannot be read off one contract: a change
that only lifts the document it was written against is overfitting, not progress. Scoring
needs a signal that works on any contract and needs no hand annotation.

The **trigger** idea comes from GRAPH-GRPO-LEX (arXiv:2511.06618): if a paragraph contains
a pattern the schema knows how to represent, the graph must contain the matching node, and
a graph that misses it is penalised. That gives recall without a gold standard.

Their extractor emits only `CLAUSE`, `DEFINED_TERM`, `PARTY` and `VALUE`, so it has nothing
to say about the deontic layer. The three triggers that matter most here — a deontic verb,
a bilateral subject, a breach cue — are ours, and so is the verbatim check.

In [1]:
import json
import re
import unicodedata
from pathlib import Path

ROOT = Path.cwd().parents[1] if Path.cwd().name == "KG" else Path.cwd()
KG_DIR = ROOT / "infra/json/kg"
PARAGRAPHS_DIR = ROOT / "infra/json/paragraphs"

STATEMENT_COLLECTIONS = ("obligations", "rights", "prohibitions")

def norm(text):
    return " ".join(unicodedata.normalize("NFKC", text or "").split())

print(ROOT, "|", len(list(KG_DIR.glob("*.json"))), "graphs")

/home/sante/Documents/FGV/me/SecondPaper | 1 graphs


## Triggers

Each pattern is a claim about the text, and each claim has a matching obligation on the
graph. The first three are the ones the prompt changes are meant to move; the last three
are the structural ones taken from the source paper.

| Pattern | What the graph must hold |
|---|---|
| deontic verb | at least one statement anchored in that paragraph |
| bilateral subject | two distinct parties bearing the burden |
| breach cue | a `Condition` anchored there |
| figure | a `Value` |
| definition formula | a `DefinedTerm` |
| citation | a `Reference` |

Regexes over-fire on recitals, which is the accepted noise floor: they are meant to catch
what is obviously present, not to judge what is arguable.

In [2]:
DEONTIC = re.compile(
    r"\b(shall|must|may not|may|agrees? to|is entitled to|reserves the right|"
    r"is not obliged|has no obligation|will not|agree to)\b",
    re.I,
)
BILATERAL = re.compile(
    r"\b(each party|both parties|neither party|either party|the parties (?:shall|agree|must)|"
    r"each hereby agree|mutually acknowledge|each other)\b",
    re.I,
)
BREACH = re.compile(
    r"\b(fails? to|failure to|default|past[-\s]due|breach(?:es|ed)?|non[-\s]?payment|"
    r"in the event that|does not (?:pay|perform|comply)|late charge|uncured|"
    r"referred to an attorney)\b",
    re.I,
)
VALUE = re.compile(
    r"(?:\$|USD|EUR|€|£)\s?\d[\d,]*(?:\.\d+)?"
    r"|\b\d+(?:\s*\d/\d)?\s*%|\b\d+\s*l?/\d\s*%"
    r"|\b(?:one|two|three|four|five|ten|thirty|sixty|ninety|\d+)\s*"
    r"(?:\(\d+\)\s*)?(?:business\s+days|days|months|years|hours)\b",
    re.I,
)
DEFINES = re.compile(
    r"[\"“][^\"”]{2,60}[\"”]\s*(?:means|shall mean)\b|\(\s*(?:the\s*)?[\"“][^\"”]{2,60}[\"”]\s*\)",
    re.I,
)
REFERENCE = re.compile(
    r"\b(?:Section|Article|Exhibit|Schedule|Appendix)\s+[A-Z0-9]|§\s*\d"
    r"|\b(?:FCRA|FDCPA|HIPAA|GDPR|ISO\s*\d+)\b",
)

## Edge typing

Adapted from their `_legal_edge`: every edge type admits only certain kinds at each end.
`grants_right_to` from anything other than a right, or into anything other than a party,
is malformed regardless of what the contract says — no text needed to catch it.

In [3]:
EDGE_RULES = {
    "is_part_of": ({"clause", "obligation", "right", "prohibition", "condition", "value", "reference"},
                   {"clause", "obligation", "right", "prohibition"}),
    "assigns_obligation_to": ({"obligation", "prohibition"}, {"party"}),
    "grants_right_to": ({"right"}, {"party"}),
    "defines": ({"clause"}, {"term"}),
    "uses": ({"clause", "obligation", "right", "prohibition"}, {"term"}),
    "references": ({"clause", "obligation", "right", "prohibition"}, {"clause"}),
    "depends_on": ({"clause", "obligation", "right", "prohibition"},
                   {"clause", "obligation", "right", "prohibition"}),
    "supersedes": ({"clause", "obligation", "right", "prohibition"},
                   {"clause", "obligation", "right", "prohibition"}),
    "modifies": ({"clause", "obligation", "right", "prohibition"},
                 {"clause", "obligation", "right", "prohibition"}),
}
DERIVED_EDGES = {"is_part_of", "assigns_obligation_to", "grants_right_to", "defines"}

COLLECTION_KIND = {
    "parties": "party", "clauses": "clause", "definedTerms": "term",
    "obligations": "obligation", "rights": "right", "prohibitions": "prohibition",
    "conditions": "condition", "references": "reference", "values": "value",
}

PHANTOM_PARTY = re.compile(
    r"^\s*(each|both|either|neither)\s+part(y|ies)\s*$|^\s*the parties\s*$", re.I
)

## The scorer

Two outputs, and only one of them is evidence.

The **indicators** are plain ratios: no tuning, comparable across contracts, and each one
traces to a countable fact about the graph.

The **score** starts at 100 and subtracts a weight per fault, capped per category so that
one bad category cannot bury the others. Those weights are a summary device, not a
measurement — nothing validates them. A score of 35 does not mean the graph is 35% right;
it means that *against the same weights*, a later run scoring 60 improved. Report the
indicators; use the score to notice movement.

In [4]:
def _pct(universe, misses):
    return 100 if not universe else round(100 * (len(universe) - len(misses)) / len(universe))

def _cap(value, limit):
    return round(min(value, limit), 1)


def score(kg, paragraphs):
    text_by_id = {p["id"]: norm(p["text"]) for p in paragraphs}
    kind_by_id = {n["id"]: kind for coll, kind in COLLECTION_KIND.items() for n in kg.get(coll, [])}
    all_ids = set(kind_by_id)

    statements = [n for coll in STATEMENT_COLLECTIONS for n in kg.get(coll, [])]
    rights = kg.get("rights", [])

    anchored = {}
    for key, nodes in (
        ("stmt", statements), ("cond", kg.get("conditions", [])), ("value", kg.get("values", [])),
        ("term", kg.get("definedTerms", [])), ("ref", kg.get("references", [])),
    ):
        bucket = anchored.setdefault(key, {})
        for n in nodes:
            for pid in n.get("paragraphIds") or []:
                bucket.setdefault(pid, []).append(n)

    def hits(pattern, minlen=50):
        return [pid for pid, t in text_by_id.items() if len(t) >= minlen and pattern.search(t)]

    misses = {}
    misses["deontic_no_statement"] = [p for p in hits(DEONTIC) if not anchored["stmt"].get(p)]
    misses["bilateral_not_split"] = [
        p for p in hits(BILATERAL)
        if anchored["stmt"].get(p)
        and len({s.get("burdenPartyId") for s in anchored["stmt"][p] if s.get("burdenPartyId")}) < 2
    ]
    misses["breach_no_condition"] = [
        p for p in hits(BREACH) if anchored["stmt"].get(p) and not anchored["cond"].get(p)
    ]
    misses["figure_no_value"] = [p for p in hits(VALUE, 20) if not anchored["value"].get(p)]
    misses["definition_no_term"] = [p for p in hits(DEFINES, 20) if not anchored["term"].get(p)]
    misses["citation_no_reference"] = [
        p for p in hits(REFERENCE, 20)
        if not anchored["ref"].get(p) and not anchored["term"].get(p)
    ]

    no_burden = [s["id"] for s in statements if not s.get("burdenPartyId")]
    rights_no_burden = [r["id"] for r in rights if not r.get("burdenPartyId")]

    phantoms = [p["id"] for p in kg.get("parties", []) if PHANTOM_PARTY.match(p.get("name") or "")]
    orphan_conditions = [
        c["id"] for c in kg.get("conditions", [])
        if not c.get("gatesId") or c["gatesId"] not in all_ids
    ]
    dangling = [
        f"{n['id']}.{field}"
        for coll, field in (
            ("obligations", "clauseId"), ("rights", "clauseId"), ("prohibitions", "clauseId"),
            ("values", "quantifiesId"), ("definedTerms", "definedInClauseId"),
            ("references", "citedById"),
        )
        for n in kg.get(coll, []) if n.get(field) and n[field] not in all_ids
    ]
    edges = kg.get("edges", [])
    illegal = [
        f"{e.get('type')}:{e.get('source')}->{e.get('target')}"
        for e in edges
        if (rule := EDGE_RULES.get(e.get("type") or "")) is None
        or kind_by_id.get(e.get("source")) not in rule[0]
        or kind_by_id.get(e.get("target")) not in rule[1]
    ]

    # `text` must be an exact substring of the paragraphs the statement came from
    not_verbatim = []
    for s in statements:
        fragment = norm(s.get("text") or "")
        source = " ".join(text_by_id.get(p, "") for p in s.get("paragraphIds") or [])
        if fragment and fragment not in source:
            not_verbatim.append(s["id"])

    n = len(statements) or 1
    model_edges = [e for e in edges if e.get("type") not in DERIVED_EDGES]

    indicators = {
        "statements": len(statements),
        "with_burden_party_%": round(100 * (len(statements) - len(no_burden)) / n),
        "rights_with_burden_%": round(100 * (len(rights) - len(rights_no_burden)) / (len(rights) or 1)),
        "bilateral_split_%": _pct(hits(BILATERAL), misses["bilateral_not_split"]),
        "breach_with_condition_%": _pct(hits(BREACH), misses["breach_no_condition"]),
        "figures_with_value_%": _pct(hits(VALUE, 20), misses["figure_no_value"]),
        "verbatim_ok_%": round(100 * (len(statements) - len(not_verbatim)) / n),
        "model_edges_per_100_statements": round(100 * len(model_edges) / n),
        "phantom_parties": len(phantoms),
        "orphan_conditions": len(orphan_conditions),
        "dangling_refs": len(dangling),
        "illegal_edges": len(illegal),
    }

    penalties = {
        "deontic_no_statement": _cap(3 * len(misses["deontic_no_statement"]), 40),
        "bilateral_not_split": _cap(3 * len(misses["bilateral_not_split"]), 20),
        "breach_no_condition": _cap(3 * len(misses["breach_no_condition"]), 20),
        "rights_no_burden": _cap(1 * len(rights_no_burden), 15),
        "statements_no_burden": _cap(0.5 * len(no_burden), 10),
        "figure_no_value": _cap(1 * len(misses["figure_no_value"]), 10),
        "not_verbatim": _cap(2 * len(not_verbatim), 20),
        "phantom_parties": _cap(10 * len(phantoms), 20),
        "orphan_conditions": _cap(2 * len(orphan_conditions), 10),
        "dangling_refs": _cap(1 * len(dangling), 10),
        "illegal_edges": _cap(1 * len(illegal), 10),
    }
    return {
        "score": round(max(0.0, 100.0 - sum(penalties.values())), 1),
        "indicators": indicators,
        "penalties": {k: v for k, v in penalties.items() if v},
        "misses": {k: v for k, v in misses.items() if v},
    }

## Run

In [5]:
def load(stem):
    kg = json.loads((KG_DIR / f"{stem}.json").read_text(encoding="utf-8"))
    paragraphs = json.loads((PARAGRAPHS_DIR / f"{stem}.json").read_text(encoding="utf-8"))["paragraphs"]
    return kg, paragraphs

results = {}
for kg_path in sorted(KG_DIR.glob("*.json")):
    if not (PARAGRAPHS_DIR / kg_path.name).exists():
        print(f"no paragraphs for {kg_path.name}, skipped")
        continue
    results[kg_path.stem] = score(*load(kg_path.stem))

for name, r in results.items():
    print(f"\n{'=' * 78}\n{name[:74]}\n  score {r['score']}/100")
    for k, v in r["indicators"].items():
        print(f"    {k:34s} {v}")
    for k, v in sorted(r["penalties"].items(), key=lambda kv: -kv[1]):
        print(f"    -{v:<5} {k}")


root_SteelVaultCorp_20081224_10-K_EX-10_16_3074935_EX-10_16_Affiliate_Agre
  score 54.5/100
    statements                         83
    with_burden_party_%                84
    rights_with_burden_%               58
    bilateral_split_%                  86
    breach_with_condition_%            50
    figures_with_value_%               80
    verbatim_ok_%                      100
    model_edges_per_100_statements     11
    phantom_parties                    0
    orphan_conditions                  0
    dangling_refs                      0
    illegal_edges                      0
    -18    deontic_no_statement
    -9     breach_no_condition
    -8     rights_no_burden
    -6.5   statements_no_burden
    -3     bilateral_not_split
    -1     figure_no_value


## What the misses look like

Reading the offending paragraphs is how the triggers stay honest. A pattern that fires on
recitals is noise to live with; a pattern that flags a real gap is the point. On the first
run over the affiliate agreement this cell is what surfaced that the one-year term and its
automatic renewal produced no statement at all — a hole in the duration requirement that
reading the graph by hand had not caught.

In [6]:
def show_misses(stem, keys=("deontic_no_statement", "bilateral_not_split", "breach_no_condition"), limit=4):
    kg, paragraphs = load(stem)
    text = {p["id"]: norm(p["text"]) for p in paragraphs}
    found = score(kg, paragraphs)["misses"]
    for key in keys:
        pids = found.get(key, [])
        print(f"\n-- {key} ({len(pids)}) --")
        for pid in pids[:limit]:
            print(f"   p{pid.split('-p-')[-1]}: {text[pid][:170]}")

if results:
    show_misses(next(iter(results)))


-- deontic_no_statement (6) --
   p8: 2. Marketing Affiliate and Equidata wish to enter into an agreement under which Marketing Affiliate may market the Services.
   p9: 3. Marketing Affiliate wishes to market the Services indirectly through third party programs, direct mail, Internet and both inbound and outbound telemarketing. In additi
   p10: Therefore, if accepted all parties agree that the following shall constitute a marketing agreement between the parties.
   p30: 6. Audit. Equidata may audit, at Equidata’s expense, the Marketing Affiliate’s marketing, practices and activities for the purpose of assuring compliance with this Agreem

-- bilateral_not_split (1) --
   p47: 10. Proprietary Information. Marketing Affiliate and Equidata mutually acknowledge that from time to time Confidential Information may be received by each. Confidential I

-- breach_no_condition (3) --
   p15: 2. Disputes. In the case of disputed charge, defined as a non-payment of an invoice for which notice o

## History

One number per run is not enough to tell a real improvement from sampling noise, and the
graph that produced a score is overwritten by the next extraction. The log keeps the
fingerprint of both inputs — the graph and the prompt that built it — so a row can always
be traced back to what produced it.

Entries are keyed by graph fingerprint: re-scoring the same graph updates nothing, only a
new extraction adds a row.

In [ ]:
import hashlib
from datetime import datetime

HISTORY = ROOT / "infra/json/score_history.json"
PROMPT_FILE = ROOT / "server/services/graph/knowledge/prompts.py"

def fingerprint(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()[:12]

def load_history():
    return json.loads(HISTORY.read_text(encoding="utf-8")) if HISTORY.exists() else {}

def prompt_version(history, sha):
    """Stable label per distinct prompt: once a fingerprint has one, it keeps it."""
    used = {}
    for runs in history.values():
        for run in runs:
            if run.get("prompt_sha"):
                used.setdefault(run["prompt_sha"], run["prompt"])
    if sha in used:
        return used[sha]
    taken = {v for v in used.values() if v.startswith("v_") and v[2:].isdigit()}
    taken |= {r["prompt"] for runs in history.values() for r in runs
              if r.get("prompt", "").startswith("v_") and r["prompt"][2:].isdigit()}
    return f"v_{max((int(v[2:]) for v in taken), default=-1) + 1}"

def record(stem, result, note=""):
    history = load_history()
    kg_sha = fingerprint(KG_DIR / f"{stem}.json")
    runs = history.setdefault(stem, [])
    if any(r["kg"] == kg_sha for r in runs):
        return False
    prompt_sha = fingerprint(PROMPT_FILE) if PROMPT_FILE.exists() else None
    entry = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "kg": kg_sha,
        "prompt": prompt_version(history, prompt_sha),
        "prompt_sha": prompt_sha,
        "score": result["score"],
        "indicators": result["indicators"],
    }
    if note:
        entry["note"] = note
    runs.append(entry)
    runs.sort(key=lambda r: r["timestamp"])
    HISTORY.write_text(
        json.dumps(dict(sorted(history.items())), ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    return True

for stem, r in results.items():
    print(("recorded" if record(stem, r) else "already logged"), "—", stem[:58])

### Iterations per contract

`prompt` is what separates an extraction change from a wording change: two rows with the
same prompt fingerprint and different scores are sampling variance, not progress.

In [ ]:
LABELS = {"score": "score", "with_burden_party_%": "burden%", "rights_with_burden_%": "rights%",
          "bilateral_split_%": "bilat%", "breach_with_condition_%": "breach%",
          "statements": "stmts"}

for document, runs in load_history().items():
    print(f"\n{document[:70]}")
    print("  " + f"{'date':>11s}{'prompt':>8s}" + "".join(f"{c:>9s}" for c in LABELS.values()))
    for r in runs:
        values = [r["score"], *(r["indicators"].get(k, "-") for k in list(LABELS)[1:])]
        print(f"  {r['timestamp'][5:]:>11s}{r['prompt']:>8s}"
              + "".join(f"{v!s:>9s}" for v in values)
              + (f"   {r['note']}" if r.get("note") else ""))